In [1]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import psycopg2
import json
from decimal import Decimal
from datetime import datetime, date, time
from uuid import UUID


class PostgresDataEncoder(json.JSONEncoder):
    """
    Custom JSON encoder to handle PostgreSQL native types.
    Converts datetime, date, time, UUID, Decimal into JSON-serializable types.
    """
    def default(self, obj):
        if isinstance(obj, (datetime, date, time)):
            return obj.isoformat()
        elif isinstance(obj, Decimal):
            return float(obj)
        elif isinstance(obj, UUID):
            return str(obj)
        return super().default(obj)


def fetch_data_from_postgres(query, db_config):
    """
    Fetches data from PostgreSQL and returns it as a JSON string.
    Native Python types are preserved (no conversion to string).
    """
    conn = None
    result_data = []

    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()

        # Execute query
        print(f"\n🔎 Executing query:\n{query}\n")
        cur.execute(query)
        rows = cur.fetchall()

        # Fetch column names
        col_names = [desc[0] for desc in cur.description]

        # Convert rows to list of dictionaries
        for row in rows:
            result_data.append(dict(zip(col_names, row)))

    except psycopg2.Error as e:
        print(f"❌ Error: {e}")
        return json.dumps({"error": str(e)}, indent=4)

    finally:
        if conn:
            conn.close()

    # Use the custom encoder here ✅
    return json.dumps(result_data, indent=4, cls=PostgresDataEncoder)


# ✅ PostgreSQL connection config
db_config = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "root",
    "port": "5432"
}

# 📥 Get the query from user
query = input("\n🛠️ Enter your SQL query: ")

# 📤 Fetch and display the result in JSON
json_result = fetch_data_from_postgres(query, db_config)
print("\n📊 Query Result in JSON format:")
print(json_result)



🔎 Executing query:
select v2tenant as sdfs from machine_data


📊 Query Result in JSON format:
[
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs": "b1d5e29f-3c3d-4649-849a-13e0e9684bee"
    },
    {
        "sdfs

In [10]:
import pandas as pd
import numpy as np
import json

def infer_dataframe_dtypes(json_result):
    """
    Takes JSON result from SQL and returns a pandas DataFrame
    with inferred and converted column data types.
    """

    # Load JSON and convert to DataFrame
    data = json.loads(json_result)
    df = pd.DataFrame(data)

    # Optional: normalize column names
    df.columns = [col.lower() for col in df.columns]

    # Define known epoch columns (ms)
    epoch_cols = ['start_timestamp', 'end_timestamp']

    # Columns to force datetime parsing from string
    datetime_cols = ['date', 'local_time']

    for col in df.columns:
        series = df[col]

        # Epoch timestamp conversion
        if col in epoch_cols and pd.api.types.is_integer_dtype(series):
            df[col] = pd.to_datetime(series.astype('Int64'), unit='ms', utc=True, errors='coerce')
            continue

        # Forced datetime conversion for known datetime-like fields
        if col in datetime_cols:
            try:
                df[col] = pd.to_datetime(series, utc=True, errors='coerce')
            except Exception as e:
                print(f"⚠️ Failed to parse datetime for '{col}': {e}")
            continue

        # General inference from values
        inferred = pd.api.types.infer_dtype(series, skipna=True)
        # print(f"🔍 Column '{col}' inferred as: {inferred}")
        if inferred in ['datetime', 'datetime64']:
            df[col] = pd.to_datetime(series, errors='coerce')

        elif inferred == 'boolean':
            df[col] = series.astype('boolean')

        elif inferred == 'integer':
            if len(str(df[col][1])) >=10:
                    df[col] = pd.to_datetime(series, utc=True, errors='coerce')
                
            else:
                df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')

        elif inferred == 'floating':
            df[col] = pd.to_numeric(series, errors='coerce', downcast='float')

        elif inferred in ['string', 'unicode']:
            # Try datetime
            # try:
            #     df[col] = pd.to_datetime(series, utc=True, errors='coerce')
            #     continue
            # except Exception:
            #     pass
            unique_ratio = series.nunique(dropna=True) / len(series)
            if unique_ratio < 0.5:
                df[col] = series.astype('category')
            else:
                df[col] = series.astype('string')

        elif inferred == 'object':
            # Try datetime
            try:
                df[col] = pd.to_datetime(series, utc=True, errors='coerce')
                continue
            except Exception:
                pass

            # Try numeric
            try:
                df[col] = pd.to_numeric(series, errors='raise')
                continue
            except Exception:
                pass

            # Try category
            unique_ratio = series.nunique(dropna=True) / len(series)
            if unique_ratio < 0.5:
                df[col] = series.astype('category')
            else:
                df[col] = series.astype('string')

    # ✅ Final datatypes
    print("\n📊 Final DataFrame dtypes:")
    print(df.dtypes)

    # Optional: return or preview
    print("\n🔍 Sample Data:")
    # print(df.head())

    # return df


In [11]:
infer_dataframe_dtypes(json_result)


📊 Final DataFrame dtypes:
sdfs    category
dtype: object

🔍 Sample Data:
